# Compare harnesses on a coding task

Give each harness its own copy of a broken calculator. Compare the fixes, real test results, answers, and timing.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/compare_harnesses/compare.ipynb)

Run these cells in order in **Google Colab**. Everything runs in its cloud runtime:
no repository checkout or laptop installation. A CPU runtime is enough.
Model calls use your provider account. Clear outputs before sharing a saved copy.

## Install and choose a model

Keep the defaults for a first run. To switch providers, change `MODEL`:
- OpenAI: `openai/gpt-5.4-mini` with `OPENAI_API_KEY`.
- Anthropic: `anthropic/claude-sonnet-4-6` with `ANTHROPIC_API_KEY`.
- OpenRouter: `openrouter/anthropic/claude-sonnet-4.6` with `OPENROUTER_API_KEY`.

LiteLLM's Python SDK handles provider translation; no gateway is required.
Optional: set `API_BASE` to your gateway URL and use its exact model alias.
Enter the gateway key when prompted, or save it as a Colab secret named
`LITELLM_API_KEY`. No environment-variable setup is required.
Leave `API_BASE` empty for direct provider access.

For Gemini, Groq, Mistral, DeepSeek, Together AI, xAI, Azure, Bedrock,
Vertex AI, or Ollama, see the [model setup guide](https://github.com/BerriAI/liteagents/blob/main/docs/models.md).
Cloud authentication and provider-specific environment variables must be configured
in this runtime before running the agent.

In [ ]:
import os

HARNESS = os.environ.get("LITEAGENTS_HARNESS", "deepagents")
MODEL = os.environ.get("LITEAGENTS_MODEL", "openai/gpt-5.4-mini")
API_BASE = os.environ.get("LITEAGENTS_API_BASE", "")  # Optional gateway URL.

HARNESSES = [HARNESS]  # Try ["deepagents", "pydantic-ai"] or any of the six below.

Available harnesses: `deepagents`, `pydantic-ai`, `claude-sdk`, `codex`,
`opencode-v1`, `opencode-v2`. Rerun the install cell after changing your selection.
Packages are reused within this runtime; a fresh Colab runtime needs its own install.
OpenCode is installed only when selected.

This preview installs from a GitHub release wheel because the PyPI name currently
belongs to another package. It does not clone the repository.

In [ ]:
# @title Install selected integrations
import shutil
import subprocess
import sys

selected_harnesses = HARNESSES
extras = sorted(set(selected_harnesses))
release = "https://github.com/BerriAI/liteagents/releases/download/v0.3.0a3"
package = f"liteagents[{','.join(extras)}] @ {release}/liteagents-0.3.0a3-py3-none-any.whl"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", package,
    "-c", f"{release}/constraints-tested.txt",
])
if any(h.startswith("opencode-") for h in selected_harnesses):
    if shutil.which("opencode") is None:
        subprocess.check_call(["npm", "install", "-g", "opencode-ai@1.18.29"])
    subprocess.check_call(["opencode", "--version"])
print("Ready:", ", ".join(selected_harnesses))

## Add your API key

In Colab, open the **key icon → Secrets**, add the key named above, and enable
notebook access. Or enter it in the hidden prompt below. The key stays out of your
code and saved outputs. If installation asks for a runtime restart,
restart once and run the cells again.

In [ ]:
# @title Connect your provider
from getpass import getpass

KEY_NAME = "LITELLM_API_KEY" if API_BASE else {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "together_ai": "TOGETHERAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "xai": "XAI_API_KEY",
    "azure": "AZURE_API_KEY",
}.get(MODEL.split("/", 1)[0])
API_KEY = None
if KEY_NAME:
    API_KEY = os.environ.get(KEY_NAME)
    if not API_KEY:
        try:
            from google.colab import userdata
        except ImportError:
            pass
        else:
            try:
                API_KEY = userdata.get(KEY_NAME)
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    API_KEY = API_KEY or getpass(f"{KEY_NAME}: ")
    if not API_KEY:
        raise ValueError(f"Provide {KEY_NAME} before running the agent.")
    os.environ[KEY_NAME] = API_KEY
else:
    print("Using provider credentials from the runtime; see the model setup guide.")
os.environ["LITEAGENTS_MODEL"] = MODEL
if API_BASE:
    os.environ["LITEAGENTS_API_BASE"] = API_BASE
MODEL_KWARGS = {"api_base": API_BASE, "api_key": API_KEY} if API_BASE else {}

## Create your profile

This is the SDK interface. The following cells change this profile to demonstrate
one feature. A temporary workspace keeps each run's files separate.

In [ ]:
import asyncio
import tempfile
from pathlib import Path
from uuid import uuid4

from liteagents import LiteAgentClient, LiteAgentOptions, ProfileOptions

profile = ProfileOptions(
    harness=HARNESS,
    model=MODEL,
    model_kwargs=MODEL_KWARGS,
    tools=[],
    max_turns=10,
    system_prompt="Use the requested tools and report their actual results. Be concise.",
)
workspace_root = Path(os.environ.get(
    "LITEAGENTS_NOTEBOOK_WORKSPACE", Path(tempfile.gettempdir()) / "liteagents-notebooks"
))
workspace_root.mkdir(parents=True, exist_ok=True)
workspace = Path(tempfile.mkdtemp(prefix="run-", dir=workspace_root)).resolve()
print("Workspace:", workspace)

## Inspect the coding task

Each harness gets its own copy of this intentionally broken calculator.
The notebook shows the SDK calls directly, then checks the resulting code with
the original tests independently of the agent's answer.

In [ ]:
from IPython.display import Code, Markdown, display

CALCULATOR = '''\
def subtract(left, right):
    return left + right
'''
TESTS = '''\
import unittest

from calculator import subtract


class SubtractionTests(unittest.TestCase):
    def test_positive(self):
        self.assertEqual(subtract(9, 4), 5)

    def test_negative(self):
        self.assertEqual(subtract(-3, 7), -10)

    def test_zero(self):
        self.assertEqual(subtract(6, 0), 6)
'''
display(Code(CALCULATOR, language="python"))
display(Code(TESTS, language="python"))

In [ ]:
PROMPT = (
    "Read calculator.py and test_calculator.py. Fix the bug in calculator.py "
    "without changing the tests, then run the tests. Summarize the change."
)
TIMEOUT_SECONDS = 180

## Run the same application with each harness

The only profile change in this loop is `harness`. To tune a harness, add its
native `harness_options` to `selected`. Timing includes startup; reported usage
comes from each harness and is not a standardized cost benchmark.

The agent can run test commands inside this runtime. Each copy is a separate directory,
not an operating-system sandbox.

In [ ]:
import difflib
import json
import time


def redact(text):
    return text.replace(API_KEY, "[REDACTED]") if API_KEY else text


profile.tools = ["read_file", "edit_file", "run_tests"]
output = workspace / f"comparison-{uuid4().hex[:8]}"
output.mkdir()
results = []
for index, harness in enumerate(HARNESSES, start=1):
    name = f"{index:02d}-{harness}"
    run_dir = output / name
    work = run_dir / "workspace"
    work.mkdir(parents=True)
    (work / "calculator.py").write_text(CALCULATOR)
    (work / "test_calculator.py").write_text(TESTS)
    selected = profile.model_copy(update={"harness": harness}, deep=True)
    report = {"name": name, "harness": harness, "status": "failed", "text": "", "usage": None}
    started = time.monotonic()
    try:
        async with asyncio.timeout(TIMEOUT_SECONDS):
            async with LiteAgentClient(
                options=LiteAgentOptions(profile=selected, cwd=work)
            ) as agent:
                run = await agent.start_run(PROMPT)
                result = await run.result()
        report.update(status="completed", text=result.text, usage=result.usage)
    except Exception as error:  # noqa: BLE001 — report each independent failure.
        report["error"] = redact(f"{type(error).__name__}: {error}")
    report["duration_seconds"] = round(time.monotonic() - started, 3)
    results.append(report)
    print(harness, report["status"], report["duration_seconds"], "seconds")

## Check the work independently

Restore the original tests before checking, so editing the tests cannot produce
a false pass. A failing harness still gets a report. Inspect the calculator diff below.

In [ ]:
for report in results:
    run_dir = output / report["name"]
    work = run_dir / "workspace"
    test_path = work / "test_calculator.py"
    report["tests_modified"] = not test_path.exists() or test_path.read_text() != TESTS
    test_path.write_text(TESTS)
    process = await asyncio.create_subprocess_exec(
        sys.executable, "-m", "unittest", "-v", cwd=work,
        stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT,
    )
    try:
        test_output, _ = await asyncio.wait_for(process.communicate(), 30)
        report["tests"] = {"exit_code": process.returncode, "output": test_output.decode()}
    except TimeoutError:
        process.kill()
        await process.wait()
        report["tests"] = {"exit_code": None, "output": "Independent tests timed out."}
    except BaseException:
        if process.returncode is None:
            process.kill()
        await process.wait()
        raise
    calculator_path = work / "calculator.py"
    updated = calculator_path.read_text() if calculator_path.exists() else ""
    diff = "".join(difflib.unified_diff(
        CALCULATOR.splitlines(True), updated.splitlines(True),
        fromfile="before/calculator.py", tofile="after/calculator.py",
    ))
    (run_dir / "changes.diff").write_text(redact(diff))
# Reports can include model output. Redact the configured key before saving.
results = json.loads(redact(json.dumps(results)))
(output / "results.json").write_text(json.dumps(results, indent=2))
rows = ["| Harness | Agent | Tests | Seconds |", "| --- | --- | --- | ---: |"]
for report in results:
    passed = report["tests"]["exit_code"] == 0 and not report["tests_modified"]
    rows.append(
        f"| {report['harness']} | {report['status']} | "
        f"{'Passed' if passed else 'Failed'} | {report['duration_seconds']} |"
    )
display(Markdown("\n".join(rows)))

In [ ]:
RESULT_INDEX = 0

In [ ]:
selected = results[RESULT_INDEX]
print("Harness:", selected["harness"], "| Status:", selected["status"])
print(selected.get("error", selected["text"]))
display(Code((output / selected["name"] / "changes.diff").read_text(), language="diff"))
print("Independent tests:", selected["tests"]["exit_code"])
print(selected["tests"]["output"])
print("Agent modified the tests:", selected["tests_modified"])
print("Native usage:", selected["usage"])

Try two harnesses with the same model, then keep the harness fixed and change `MODEL`.
Each run's files and `results.json` are in `output`. Download anything you want to keep
from Colab's Files sidebar before the runtime resets.